In [2]:
# Создаёт ~/mlproject/src/__init__.py и ~/mlproject/src/predictor.py
import os
from pathlib import Path

PROJECT = Path(os.path.expanduser("~/mlproject"))
SRC = PROJECT / "src"
SRC.mkdir(parents=True, exist_ok=True)

INIT_CODE = r'''"""Пакет src проекта LR2: валидация входа и предсказание базовой модели."""
from .predictor import predict, predict_from_file, PredictorError  # noqa: F401
'''

PREDICTOR_CODE = r'''"""
Единая точка входа для инференса базовой span-type модели (ЛР2).

Контракт:
  predict(spans) -> list[dict]
  predict_from_file(path) -> list[dict]

Порядок: валидация -> предобработка (TF-IDF) -> проверка формы -> модель.
Любое отклонение входа -> PredictorError (наследники: ValidationError,
InputTooLargeError, FormatError, ModelError).
"""
from __future__ import annotations

import os
import re
import unicodedata
from pathlib import Path

import joblib
import numpy as np

# ==== ОГРАНИЧЕНИЯ ====
MAX_SPANS_PER_REQUEST = 1_000
MAX_SPAN_LEN_CHARS    = 1_000
MAX_TOTAL_CHARS       = 1_000_000
MAX_FILE_BYTES        = 10 * 1024 * 1024

_CTRL_RE = re.compile(r"[\x00-\x08\x0b\x0c\x0e-\x1f\x7f]")
_BINARY_MAGIC = (
    b"%PDF", b"PK\x03\x04", b"\x89PNG",
    b"\xff\xd8\xff", b"GIF8", b"\x1f\x8b",
)
_DEFAULT_ART = Path(os.path.expanduser("~/mlproject/artifacts"))


class PredictorError(Exception):
    code = "predictor_error"

class ValidationError(PredictorError):
    code = "validation_error"

class InputTooLargeError(PredictorError):
    code = "input_too_large"

class FormatError(PredictorError):
    code = "format_error"

class ModelError(PredictorError):
    code = "model_error"


_VECTORIZER = None
_MODEL = None
_EXPECTED_N_FEATURES = None


def load_model(artifacts_dir=_DEFAULT_ART):
    art = Path(artifacts_dir)
    vec_path = art / "lab2_span_tfidf.joblib"
    mdl_path = art / "lab2_span_logreg.joblib"
    if not vec_path.exists() or not mdl_path.exists():
        raise ModelError(
            f"Артефакты не найдены в {art}. "
            f"Запустите lab2_baseline_model.ipynb."
        )
    return joblib.load(vec_path), joblib.load(mdl_path)


def _ensure_model():
    global _VECTORIZER, _MODEL, _EXPECTED_N_FEATURES
    if _MODEL is not None:
        return
    _VECTORIZER, _MODEL = load_model()
    vec_n = _VECTORIZER.transform(["x"]).shape[1]
    model_n = getattr(_MODEL, "n_features_in_", None)
    if model_n is not None and model_n != vec_n:
        raise ModelError(
            f"Несовместимость артефактов: model n_features_in_={model_n}, "
            f"vectorizer n_features={vec_n}."
        )
    _EXPECTED_N_FEATURES = vec_n


def _check_string(s, field, max_len):
    if not isinstance(s, str):
        raise ValidationError(f"{field}: ожидается str, получено {type(s).__name__}")
    if s == "":
        raise ValidationError(f"{field}: пустая строка")
    if "\x00" in s:
        raise ValidationError(f"{field}: содержит NUL-байт")
    if _CTRL_RE.search(s):
        raise ValidationError(f"{field}: содержит управляющие символы")
    if len(s) > max_len:
        raise InputTooLargeError(f"{field}: длина {len(s)} > {max_len}")
    return s


def _normalize(s):
    return unicodedata.normalize("NFKC", s).strip()


def validate_input(spans):
    """Возвращает нормализованный list[str]. NaN/Inf/None -> ValidationError."""
    try:
        import pandas as pd
        if isinstance(spans, pd.DataFrame):
            for col in ("span", "text"):
                if col in spans.columns:
                    spans = spans[col].tolist()
                    break
            else:
                raise ValidationError(
                    f"DataFrame: нужна колонка 'span' или 'text', "
                    f"есть {list(spans.columns)}"
                )
    except ImportError:
        pass

    if spans is None:
        raise ValidationError("input: None не допускается")
    if isinstance(spans, str):
        spans = [spans]
    if not isinstance(spans, (list, tuple)):
        raise ValidationError(
            f"input: ожидается list[str] или str, получено {type(spans).__name__}"
        )
    if len(spans) == 0:
        raise ValidationError("input: пустой список")
    if len(spans) > MAX_SPANS_PER_REQUEST:
        raise InputTooLargeError(
            f"input: {len(spans)} спанов > лимита {MAX_SPANS_PER_REQUEST}"
        )

    cleaned = []
    total_chars = 0
    for i, s in enumerate(spans):
        if isinstance(s, float):
            if np.isnan(s):
                raise ValidationError(f"input[{i}]: NaN не допускается")
            if np.isinf(s):
                raise ValidationError(f"input[{i}]: Inf не допускается")
            raise ValidationError(f"input[{i}]: ожидается str, получено float")
        s = _check_string(s, f"input[{i}]", MAX_SPAN_LEN_CHARS)
        s = _normalize(s)
        if not s:
            raise ValidationError(f"input[{i}]: пустая строка после нормализации")
        total_chars += len(s)
        cleaned.append(s)

    if total_chars > MAX_TOTAL_CHARS:
        raise InputTooLargeError(
            f"input: суммарно {total_chars} символов > лимита {MAX_TOTAL_CHARS}"
        )
    return cleaned


def predict(spans, *, top_k=None):
    """Предсказать тип сущности для каждого спана."""
    _ensure_model()
    cleaned = validate_input(spans)

    try:
        X = _VECTORIZER.transform(cleaned)
    except Exception as e:
        raise PredictorError(f"preprocessing: {e}") from e

    if X.shape[0] != len(cleaned):
        raise ModelError(f"shape: {X.shape[0]} != {len(cleaned)}")
    if X.shape[1] != _EXPECTED_N_FEATURES:
        raise ModelError(
            f"shape: {X.shape[1]} признаков != ожидаемых {_EXPECTED_N_FEATURES}"
        )

    try:
        proba = _MODEL.predict_proba(X)
    except Exception as e:
        raise ModelError(f"inference: {e}") from e

    classes = _MODEL.classes_
    out = []
    for i, span in enumerate(cleaned):
        p = proba[i]
        k = len(classes) if top_k is None else min(top_k, len(classes))
        order = np.argsort(p)[::-1][:k]
        preds = [{"label": str(classes[j]), "confidence": float(p[j])} for j in order]
        item = {"span": span, "label": preds[0]["label"],
                "confidence": preds[0]["confidence"]}
        if top_k is not None:
            item["top_k"] = preds
        out.append(item)
    return out


_ALLOWED_EXT = {".txt"}


def predict_from_file(path, *, top_k=None):
    """Читает txt (один спан на строку), валидирует и предсказывает."""
    p = Path(path)

    if not p.exists():
        raise ValidationError(f"file: {p} не существует")
    if not p.is_file():
        raise ValidationError(f"file: {p} — не обычный файл")

    st = p.lstat()
    if st.st_size > MAX_FILE_BYTES:
        raise InputTooLargeError(
            f"file: {st.st_size} байт > лимита {MAX_FILE_BYTES}"
        )
    if p.suffix.lower() not in _ALLOWED_EXT:
        raise FormatError(
            f"file: расширение {p.suffix!r} не разрешено ({sorted(_ALLOWED_EXT)})"
        )

    with open(p, "rb") as f:
        head = f.read(8 * 1024)
        for magic in _BINARY_MAGIC:
            if head.startswith(magic):
                raise FormatError(
                    f"file: бинарная сигнатура {magic!r} — не текст"
                )
        if b"\x00" in head:
            raise FormatError("file: NUL-байт в начале — похоже на бинарник")
        f.seek(0)
        raw = f.read()

    if len(raw) > MAX_FILE_BYTES:
        raise InputTooLargeError(
            f"file: после чтения {len(raw)} байт > лимита {MAX_FILE_BYTES}"
        )

    text = None
    for enc in ("utf-8", "utf-8-sig"):
        try:
            text = raw.decode(enc)
            break
        except UnicodeDecodeError:
            continue
    if text is None:
        raise FormatError("file: не удалось декодировать как UTF-8")

    spans = [ln for ln in (l.strip() for l in text.splitlines()) if ln]
    return predict(spans, top_k=top_k)
'''

(SRC / "__init__.py").write_text(INIT_CODE, encoding="utf-8")
(SRC / "predictor.py").write_text(PREDICTOR_CODE, encoding="utf-8")
print(f"OK: {SRC}/__init__.py ({len(INIT_CODE)} байт)")
print(f"OK: {SRC}/predictor.py ({len(PREDICTOR_CODE)} байт)")

OK: /home/mluser/mlproject/src/__init__.py (156 байт)
OK: /home/mluser/mlproject/src/predictor.py (7508 байт)


In [3]:
import os, sys, importlib

sys.path.insert(0, os.path.expanduser("~/mlproject"))

# перезагрузка на случай правок
if "src.predictor" in sys.modules:
    importlib.reload(sys.modules["src.predictor"])
import src.predictor as pred_mod
importlib.reload(pred_mod)

from src.predictor import (
    predict, predict_from_file, PredictorError,
    ValidationError, InputTooLargeError, FormatError, ModelError,
    MAX_SPANS_PER_REQUEST, MAX_SPAN_LEN_CHARS, MAX_TOTAL_CHARS, MAX_FILE_BYTES,
)

print("Модуль загружен")
print(f"Лимиты: spans<={MAX_SPANS_PER_REQUEST}, "
      f"span_len<={MAX_SPAN_LEN_CHARS}, "
      f"total_chars<={MAX_TOTAL_CHARS}, "
      f"file_bytes<={MAX_FILE_BYTES}")

Модуль загружен
Лимиты: spans<=1000, span_len<=1000, total_chars<=1000000, file_bytes<=10485760


In [6]:
print("=== Позитивный вызов predict() ===")
result = predict([
    "ivan.petrov@example.com",
    "+7 900 123-45-67",
    "4276 1234 5678 9012",
    "123-456-789 00",
])
for r in result:
    print(f"{r['label']:<20} conf={r['confidence']:.3f}  {r['span']!r}")

# top_k передаём явно при вызове
print("\n=== top_k=3 для первого спана ===")
result_top3 = predict(["ivan.petrov@example.com"], top_k=3)
for p in result_top3[0]["top_k"]:
    print(f"  {p['label']:<20} {p['confidence']:.3f}")

=== Позитивный вызов predict() ===
email                conf=0.921  'ivan.petrov@example.com'
mobile_phone         conf=0.917  '+7 900 123-45-67'
tin                  conf=0.326  '4276 1234 5678 9012'
snils                conf=0.679  '123-456-789 00'

=== top_k=3 для первого спана ===
  email                0.921
  address_region       0.008
  address_city         0.008


In [7]:
print("=== Негативные тесты predict() ===\n")

cases = [
    ("None",              None),
    ("пустой список",     []),
    ("[None]",            [None]),
    ("[NaN]",             [float("nan")]),
    ("[Inf]",             [float("inf")]),
    ("не список",         42),
    ("длинный спан",      "x" * 2000),
    ("NUL-байт",          ["ok", "\x00"]),
    ("управляющий символ",["ok", "a\x01b"]),
    ("слишком много",     [f"span_{i}" for i in range(2000)]),
]

for name, case in cases:
    try:
        predict(case)
        print(f"❌ {name}: НЕ отклонено (это баг)")
    except PredictorError as e:
        print(f"✅ {name}: [{e.code}] {str(e)[:90]}")

=== Негативные тесты predict() ===

✅ None: [validation_error] input: None не допускается
✅ пустой список: [validation_error] input: пустой список
✅ [None]: [validation_error] input[0]: ожидается str, получено NoneType
✅ [NaN]: [validation_error] input[0]: NaN не допускается
✅ [Inf]: [validation_error] input[0]: Inf не допускается
✅ не список: [validation_error] input: ожидается list[str] или str, получено int
✅ длинный спан: [input_too_large] input[0]: длина 2000 > 1000
✅ NUL-байт: [validation_error] input[1]: содержит NUL-байт
✅ управляющий символ: [validation_error] input[1]: содержит управляющие символы
✅ слишком много: [input_too_large] input: 2000 спанов > лимита 1000
